# prepare_eastream_inputs

Run **cell 2** (definitions), then **cell 3** (execute + verify outputs).
Outputs: `Results/forcing/rainplusmelt.csv`, `Results/forcing/PET.csv`.

In [ ]:
# Optional setup cell (cell 3 calls main(), which sets project root automatically).
pass


In [ ]:
import argparse
import os
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import savemat

# Plotting is optional but recommended for checks
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------------------
# Basin IDs we want to use (must match the IDs in the CSV file names
# and in estreams_hydrometeo_signatures.csv)
BASIN_IDS = ["CH000127", "DESN1585", "FR002785","GB000215","SE000197", "ES000700"]

# Paths to input files (relative to your project root)
CSV_ESTREAM_DIR = "data/csv_estream"
METEO_TEMPLATE = f"{CSV_ESTREAM_DIR}/estreams_meteorology_{{basin}}.csv"
SIGNATURES_PATH = f"{CSV_ESTREAM_DIR}/estreams_hydrometeo_signatures.csv"

# Degree-day factor (mm / °C / day) based on Singh et al., 2000
DDF = 5.9

MATLAB_DIR = "Matlab_files"
FORCING_CSV_DIR = "Results/forcing"
DIAG_DIR = "Results/diagnostics"
PLOT_MAX_DAYS = 1500  # plotting all ~27k daily bars is very slow; plot a window
SAVE_DIAGNOSTIC_PLOTS = True  # set False in notebooks if plots hang; CSVs still save


def find_project_root() -> Path:
    """Locate thesis project root (works from repo root or notebooks/)."""
    start = Path.cwd().resolve()
    candidates = [start]
    if start.name == "notebooks":
        candidates.append(start.parent)
    for path in candidates + list(start.parents):
        marker = path / "data" / "csv_estream" / "estreams_hydrometeo_signatures.csv"
        if marker.exists():
            return path
    raise FileNotFoundError(
        "Could not find project root (missing data/csv_estream/estreams_hydrometeo_signatures.csv). "
        f"Current working directory: {start}"
    )


# ---------------------------------------------------------------------
# HELPER FUNCTIONS
# ---------------------------------------------------------------------
def get_frac_snow_per_basin(signatures_path: str, basin_ids: list[str]) -> dict:
    """
    Read estreams_hydrometeo_signatures.csv and return a dict:
    { basin_id: frac_snow_value }
    """
    # Different downloads of the eStreams signatures file may be comma- or semicolon-
    # separated, and sometimes the basin id column comes in as an unnamed first column.
    # Read with delimiter auto-detection and then normalize headers.
    sig = pd.read_csv(signatures_path, sep=None, engine="python")
    sig.columns = [str(c).strip().lstrip("\ufeff") for c in sig.columns]
    if "basin_id" not in sig.columns:
        unnamed_first = next((c for c in sig.columns if c.lower().startswith("unnamed")), None)
        if unnamed_first is not None:
            sig = sig.rename(columns={unnamed_first: "basin_id"})

    frac_dict = {}
    for b in basin_ids:
        row = sig.loc[sig["basin_id"].astype(str).str.strip() == b]
        if row.empty:
            raise ValueError(f"basin_id {b} not found in signatures file.") # in E-Streams dataset 2025 some csv files are missing the basin_id name, just add it manually
        frac_snow = float(row["frac_snow"].iloc[0])
        frac_dict[b] = frac_snow
    return frac_dict


def compute_rainplusmelt_components_for_basin(
    df: pd.DataFrame, frac_snow: float
) -> dict[str, np.ndarray]:
    """
    Compute rainplusmelt and its components for one basin, given:
    - df: dataframe with columns 'p_mean', 't_mean'
    - frac_snow: scalar from signatures (0..1)

    Rules (as you described):
    - If T > 0°C: all P is liquid (Pliq = P), no new snow.
    - If T <= 0°C: snow_precip = frac_snow * P, rain_precip = (1-frac_snow)*P.
    - Snowpack_t = snowpack_{t-1} + snow_precip - melt
    - Melt = T * P * DDF  (we cap melt so it can’t be larger than the snowpack)
    - rainplusmelt = liquid_precip + melt
    """
    p = df["p_mean"].to_numpy(dtype=float)     # daily precipitation
    t = df["t_mean"].to_numpy(dtype=float)     # daily mean temperature

    n = len(p)
    rainplusmelt = np.zeros(n, dtype=float)
    rain_precip = np.zeros(n, dtype=float)
    snow_precip = np.zeros(n, dtype=float)
    melt = np.zeros(n, dtype=float)
    snowpack_ts = np.zeros(n, dtype=float)

    snowpack = 0.0

    for i in range(n):
        P = p[i]
        T = t[i]

        if T > 0.0:
            # All precipitation is liquid
            snow_p = 0.0
            rain_p = P
        else:
            # Split into snow and liquid using frac_snow
            snow_p = frac_snow * P
            rain_p = (1.0 - frac_snow) * P

        # Update snowpack with new snow
        snowpack += snow_p

        # Compute melt (following your description: Melt = T * P * DDF)
        if T > 0.0 and snowpack > 0.0:
            m = T * P * DDF
            # Melt cannot exceed current snowpack
            m = min(m, snowpack)
        else:
            m = 0.0

        snowpack -= m
        if snowpack < 0.0:
            snowpack = 0.0

        # Rainplusmelt = liquid precipitation + snowmelt
        rainplusmelt[i] = rain_p + m

        rain_precip[i] = rain_p
        snow_precip[i] = snow_p
        melt[i] = m
        snowpack_ts[i] = snowpack

    return {
        "rainplusmelt": rainplusmelt,
        "rain_precip": rain_precip,
        "snow_precip": snow_precip,
        "melt": melt,
        "snowpack": snowpack_ts,
    }


def plot_rainplusmelt_diagnostics(
    basin_id: str,
    dates: pd.Series,
    p: np.ndarray,
    t: np.ndarray,
    frac_snow: float,
    components: dict[str, np.ndarray],
    out_png_path: str,
) -> None:
    """Create a diagnostic plot showing how rainplusmelt was generated."""
    # Plot only a window to keep rendering fast and readable
    if len(dates) > PLOT_MAX_DAYS:
        dates = dates.iloc[:PLOT_MAX_DAYS]
        p = p[:PLOT_MAX_DAYS]
        t = t[:PLOT_MAX_DAYS]
        components = {k: v[:PLOT_MAX_DAYS] for k, v in components.items()}

    rpm = components["rainplusmelt"]
    rain_p = components["rain_precip"]
    snow_p = components["snow_precip"]
    melt = components["melt"]
    snowpack = components["snowpack"]

    fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)

    # 1) Forcing + split
    ax = axes[0]
    ax.plot(dates, p, color="gray", linewidth=1, label="P total")
    ax.fill_between(dates, 0, rain_p, color="dodgerblue", alpha=0.6, label="Rain part")
    ax.fill_between(dates, rain_p, rain_p + snow_p, color="slateblue", alpha=0.6, label="Snow part")
    ax.set_ylabel("mm/day")
    ax.set_title(f"{basin_id} – Precip split (first {len(dates)} days, frac_snow={frac_snow:.3f})")
    ax.legend(ncol=3, loc="upper right")

    # 2) Temperature (melt condition)
    ax = axes[1]
    ax.plot(dates, t, color="tomato", label="T mean")
    ax.axhline(0.0, color="black", linewidth=1)
    ax.set_ylabel("°C")
    ax.set_title("Temperature (0°C threshold)")
    ax.legend(loc="upper right")

    # 3) Snowpack + melt
    ax = axes[2]
    ax.plot(dates, snowpack, color="black", label="Snowpack")
    ax.fill_between(dates, 0, melt, color="orange", alpha=0.4, label="Melt")
    ax.set_ylabel("mm")
    ax.set_title("Snowpack and melt")
    ax.legend(loc="upper right")

    # 4) Final rainplusmelt
    ax = axes[3]
    ax.plot(dates, rpm, color="navy", label="Rainplusmelt")
    ax.set_ylabel("mm/day")
    ax.set_title("Rainplusmelt = rain + melt")
    ax.legend(loc="upper right")

    fig.tight_layout()
    fig.savefig(out_png_path, dpi=150)
    plt.close(fig)


# ---------------------------------------------------------------------
# MAIN SCRIPT
# ---------------------------------------------------------------------
def parse_prepare_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description="Build rainplusmelt/PET from eStreams per-basin meteorology CSVs.")
    p.add_argument(
        "--basins",
        default="",
        help="Comma-separated basin IDs. If empty, uses BASIN_IDS in this script (default six-basin list).",
    )
    return p.parse_known_args()[0]


def main(basin_ids: list[str] | None = None, *, save_diagnostic_plots: bool = SAVE_DIAGNOSTIC_PLOTS) -> None:
    if basin_ids is None:
        args = parse_prepare_args()
        basin_ids = (
            [b.strip() for b in args.basins.split(",") if b.strip()]
            if args.basins.strip()
            else list(BASIN_IDS)
        )

    root = find_project_root()
    os.chdir(root)
    print(f"Project root: {root.resolve()}")

    for d in (MATLAB_DIR, FORCING_CSV_DIR, DIAG_DIR):
        os.makedirs(d, exist_ok=True)

    frac_snow_dict = get_frac_snow_per_basin(SIGNATURES_PATH, basin_ids)

    rainplusmelt_list: list[np.ndarray] = []
    pet_list: list[np.ndarray] = []
    date_index = None
    basin_records: list[dict] = []

    for basin in basin_ids:
        print(f"Processing basin {basin}...", flush=True)
        meteo_path = METEO_TEMPLATE.format(basin=basin)
        if not Path(meteo_path).exists():
            raise FileNotFoundError(f"Missing meteorology file: {Path(meteo_path).resolve()}")

        df = pd.read_csv(meteo_path)
        df["date"] = pd.to_datetime(df["date"])

        if date_index is None:
            date_index = df["date"]
        elif not np.array_equal(date_index.to_numpy(), df["date"].to_numpy()):
            raise ValueError(f"Dates for basin {basin} do not match the first basin.")

        pet = df["pet_mean"].to_numpy(dtype=float)
        frac_snow = frac_snow_dict[basin]
        components = compute_rainplusmelt_components_for_basin(df, frac_snow=frac_snow)

        rainplusmelt_list.append(components["rainplusmelt"])
        pet_list.append(pet)
        basin_records.append(
            {
                "basin": basin,
                "df": df,
                "frac_snow": frac_snow,
                "components": components,
            }
        )

    rainplusmelt = np.column_stack(rainplusmelt_list)
    PET = np.column_stack(pet_list)

    print("Shapes:")
    print("  rainplusmelt:", rainplusmelt.shape)
    print("  PET:", PET.shape)

    # Save forcing outputs first (before optional plots).
    rpm_csv = Path(FORCING_CSV_DIR) / "rainplusmelt.csv"
    pet_csv = Path(FORCING_CSV_DIR) / "PET.csv"
    out_rpm = pd.DataFrame({"date": date_index})
    out_pet = pd.DataFrame({"date": date_index})
    for j, basin in enumerate(basin_ids):
        out_rpm[basin] = rainplusmelt[:, j]
        out_pet[basin] = PET[:, j]
    out_rpm.to_csv(rpm_csv, index=False)
    out_pet.to_csv(pet_csv, index=False)
    print(f"Wrote {rpm_csv.resolve()}")
    print(f"Wrote {pet_csv.resolve()}")

    savemat(str(Path(MATLAB_DIR) / "rainplusmelt.mat"), {"rainplusmelt": rainplusmelt})
    savemat(str(Path(MATLAB_DIR) / "PET.mat"), {"PET": PET})
    print(f"Wrote {(Path(MATLAB_DIR) / 'rainplusmelt.mat').resolve()}")
    print(f"Wrote {(Path(MATLAB_DIR) / 'PET.mat').resolve()}")

    for rec in basin_records:
        basin = rec["basin"]
        df = rec["df"]
        components = rec["components"]
        frac_snow = rec["frac_snow"]
        diag = pd.DataFrame(
            {
                "date": df["date"],
                "p_mean": df["p_mean"].astype(float),
                "t_mean": df["t_mean"].astype(float),
                "pet_mean": df["pet_mean"].astype(float),
                "frac_snow_scalar": frac_snow,
                "rain_precip": components["rain_precip"],
                "snow_precip": components["snow_precip"],
                "melt": components["melt"],
                "snowpack": components["snowpack"],
                "rainplusmelt": components["rainplusmelt"],
            }
        )
        diag.to_csv(Path(DIAG_DIR) / f"diagnostics_{basin}.csv", index=False)

        if save_diagnostic_plots:
            plot_rainplusmelt_diagnostics(
                basin_id=basin,
                dates=df["date"],
                p=df["p_mean"].to_numpy(dtype=float),
                t=df["t_mean"].to_numpy(dtype=float),
                frac_snow=frac_snow,
                components=components,
                out_png_path=str(Path(DIAG_DIR) / f"diagnostics_{basin}.png"),
            )

    print(f"Diagnostics -> {Path(DIAG_DIR).resolve()}/")


In [ ]:
# Run pipeline and verify output files exist
main(save_diagnostic_plots=True)

from pathlib import Path
for rel in ("Results/forcing/rainplusmelt.csv", "Results/forcing/PET.csv"):
    p = Path(rel).resolve()
    if p.exists():
        print(f"OK  {p}  ({p.stat().st_size:,} bytes)")
    else:
        print(f"MISSING  {p}")
